# CafChem GPT — Foundation training on Colab (GPU)

This notebook clones the `SMILES_GPT` repo, picks up the foundation-model
training from the checkpoint we already built locally
(`data/GPT_ZN305_pytorch.pt`, currently 10 epochs / loss ~0.67), trains more
epochs on a Colab T4 GPU, and lets you download the updated `.pt` back to your
machine.

**Why this works:** the existing PyTorch refactor already supports resuming —
set `RESUME_FOUNDATION=1` and `run_gpt.py` loads the checkpoint and continues.
The 402 MB `ZN305K_tokenized.npz` cache is gitignored, so step 4 rebuilds it on
Colab (a few minutes, once).

Cells are meant to run top-to-bottom.

## 1. Clone the repo

If the repo is **private**, paste a GitHub Personal Access Token (PAT) with
`repo` scope into `GITHUB_TOKEN` below. If it's public, leave it blank.

Create a PAT at: https://github.com/settings/tokens

In [ ]:
#@title Clone SMILES_GPT
import os, shutil, subprocess, sys

GITHUB_TOKEN = ""  #@param {type:"string"}
REPO_URL    = "https://github.com/MauricioCafiero/SMILES_GPT.git"  #@param {type:"string"}
REPO_DIR    = "SMILES_GPT"  #@param {type:"string"}

# Wipe a stale clone so re-runs are clean.
if os.path.isdir(REPO_DIR):
    shutil.rmtree(REPO_DIR)

clone_url = REPO_URL
if GITHUB_TOKEN:
    # Insert token into https URL: https://<token>@github.com/...
    if "@github.com" not in REPO_URL:
        clone_url = REPO_URL.replace("https://github.com",
                                     f"https://{GITHUB_TOKEN}@github.com")

r = subprocess.run(["git", "clone", clone_url, REPO_DIR],
                  capture_output=True, text=True)
print(r.stdout)
if r.returncode != 0:
    print("CLONE FAILED:\n", r.stderr)
    raise SystemExit(1)

# Work from the repo root so all relative paths (data/, code/) resolve.
os.chdir(REPO_DIR)
print("Cwd:", os.getcwd())
print("Files:", os.listdir("."))

## 2. Install dependencies

Colab already ships `torch` (CUDA build), `pandas`, `numpy`, `Pillow`.
We only need to add `rdkit` (imported by `CafChemGPT.py`).

In [ ]:
#@title pip install rdkit
import subprocess, sys
subprocess.check_call([sys.executable, "-m", "pip", "install",
                        "rdkit", "-q"])
import rdkit
print("rdkit", rdkit.__version__)

## 3. Verify the GPU and the saved checkpoint

On Colab the default device picker (`get_device`) will pick CUDA. We confirm a
T4 (or better) is attached and that the foundation checkpoint we'll resume
from is present.

In [ ]:
import torch, os
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("VRAM:", round(torch.cuda.get_device_properties(0).total_memory/1e9, 1), "GB")
else:
    print("\u26a0 No GPU — go to Runtime > Change runtime type > T4 GPU.")

ckpt = "data/GPT_ZN305_pytorch.pt"
print(f"\nFoundation checkpoint {ckpt}: "
      f"{'present' if os.path.exists(ckpt) else 'MISSING'}")
if os.path.exists(ckpt):
    print("  size:", round(os.path.getsize(ckpt)/1e6, 1), "MB")
    cfg = torch.load(ckpt, map_location="cpu", weights_only=False)
    print("  config:", cfg.get("config", cfg))

## 4. (Re)build the tokenized dataset cache

The `ZN305K_tokenized.npz` cache (402 MB) is gitignored and is **not** in the
repo, so we rebuild it once from `data/ZN305K_smiles.csv`. This takes a few
minutes.

If you'd rather not wait, you can instead upload your local `ZN305K_tokenized.npz`
to `data/` (use the Files panel on the left) and skip this cell — but rebuilding
is usually simpler than a 400 MB upload.

In [ ]:
import os, sys, numpy as np
sys.path.insert(0, "code")

from CafChemGPT import make_datasets, get_device
print("Device:", get_device())

cache = "data/ZN305K_tokenized.npz"
if os.path.exists(cache):
    z = np.load(cache)
    print(f"Cache already present: fx {z['fx'].shape}, fy {z['fy'].shape}, "
          f"VOCAB {int(z['VOCAB_SIZE'])}, max_len {int(z['max_length'])}")
else:
    print("Rebuilding tokenized cache from data/ZN305K_smiles.csv ...")
    fx, fy, VOCAB_SIZE, tokenizer, max_length = make_datasets(
        "data/ZN305K_smiles.csv", "SMILES")
    np.savez(cache, fx=fx, fy=fy, VOCAB_SIZE=VOCAB_SIZE, max_length=max_length)
    print(f"Saved {cache}: fx {fx.shape}, fy {fy.shape}, "
          f"VOCAB {VOCAB_SIZE}, max_len {max_length}")

## 5. Resume foundation training

This loads the existing checkpoint and trains `EXTRA_EPOCHS` more. Each epoch
on a T4 with `batch_size=512` over ~305K samples takes roughly 30–60 s, so the
defaults below (20 extra epochs, ~10–20 min) are a safe first run — bump them
up once you confirm it's training.

The trained model is saved back to `data/GPT_ZN305_pytorch.pt` (overwriting the
old checkpoint, so download a copy first if you want to keep the 10-epoch
version).

In [ ]:
#@title Train more epochs on the foundation model
import os, sys, numpy as np
sys.path.insert(0, "code")

EXTRA_EPOCHS = 20   #@param {type:"integer"}
BATCH_SIZE   = 512  #@param {type:"integer"}
LR           = 1e-3 #@param {type:"number"}

from CafChemGPT import load_gpt, train_gpt, save_gpt, get_device

device = get_device()
cache = "data/ZN305K_tokenized.npz"
z = np.load(cache)
fx, fy = z["fx"], z["fy"]
VOCAB_SIZE, max_length = int(z["VOCAB_SIZE"]), int(z["max_length"])
print(f"fx {fx.shape} | fy {fy.shape} | VOCAB {VOCAB_SIZE} | max_len {max_length}")
print("Device:", device)

FOUNDATION_FILE = "data/GPT_ZN305_pytorch"  # .pt appended by load/save
# total_layers=2 matches the original foundation architecture.
gpt = load_gpt(FOUNDATION_FILE, total_layers=2,
               max_length=max_length, VOCAB_SIZE=VOCAB_SIZE)

# Train EXTRA_EPOCHS more, resuming from the loaded weights.
gpt = train_gpt(gpt, fx, fy, epochs=EXTRA_EPOCHS, batch_size=BATCH_SIZE, lr=LR)

# Overwrite the checkpoint with the newly trained weights.
save_gpt(gpt, FOUNDATION_FILE)
print(f"\nFoundation checkpoint updated -> {FOUNDATION_FILE}.pt")

## 6. Download the updated `.pt` back to your Mac

Run this cell to push the checkpoint to your browser download. After it lands,
drop it into the local repo's `data/` folder (replacing the old one) and commit
if you want it versioned. The file is small (~3–4 MB).

Tip: also grab the layer-store text file next to it for completeness.

In [ ]:
from google.colab import files
import os

pt_path = "data/GPT_ZN305_pytorch.pt"
print(f"Downloading {pt_path} ({round(os.path.getsize(pt_path)/1e6,1)} MB) ...")
files.download(pt_path)

# Optional: also pull the layer-name store written by save_gpt.
ls_path = "data/layer_store_GPT_ZN305_pytorch.txt"
if os.path.exists(ls_path):
    files.download(ls_path)
    print("Also downloading", ls_path)

## 7. (Optional) Quick generation sanity check

Generate a few molecules from the freshly trained foundation to confirm it
produces valid SMILES. The more epochs trained, the fewer invalid molecules
you should see.

In [ ]:
import sys; sys.path.insert(0, "code")
from CafChemGPT import load_gpt, make_prompts, gen_mols
from smiles_tokenizer import SmilesTokenizer

tok = SmilesTokenizer(vocab_file="data/vocab_305K.txt")
VOCAB = tok.vocab_size()
model = load_gpt("data/GPT_ZN305_pytorch", 2, 166, VOCAB)

prompts = make_prompts(8, 2)
pic, smiles = gen_mols(prompts, use_ramp=True, model=model, tokenizer=tok,
                       TEMP=1.5, VOCAB_SIZE=VOCAB)
pic.save("generated_molecules.png")
print(f"{len(smiles)} unique molecules. Samples:", smiles[:5])
from IPython.display import display, Image
display(Image("generated_molecules.png"))